#Loading Dataset

##Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Set Dataset path

In [2]:
train_dataset = '/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/datasets/processed_files/wolfset_train.csv'
test_dataset = '/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/datasets/processed_files/wolfset_test.csv'

##Load Datasets as Pandas Dataframe

In [3]:
import pandas as pd
train_df = pd.read_csv(train_dataset)
train_df.head()

,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,spectral_rolloff,label,motor_code,transient_code,noise_code,filename
0,0.050365,0.009733,-1.055566,0.316141,0.669253,0.655374,-0.136285,0.004763,-1.181118,-0.990770,-0.350304,-0.880928,-0.515005,-0.626708,2,2000,4,0,A02000T04N00.wav
1,0.763096,-0.969426,-0.897181,-1.729356,-1.666303,-0.409425,1.724379,0.146817,0.466983,-0.349870,0.631503,0.174636,-0.639133,0.982519,1,20000,2,0,A20000T02N00.wav
2,0.761177,-0.606739,-0.410336,-0.881375,-0.883765,-1.129219,-0.685946,-1.338750,-0.162373,-0.273524,0.268508,0.113154,1.561239,0.788899,1,20000,0,4,A20000T00N04x02.wav
3,-0.258524,0.345973,1.287845,0.868921,-0.119996,1.767695,1.013693,0.791222,0.545691,-1.498075,-0.502849,0.491811,-1.149545,0.169105,4,10,0,0,A00010T00N00.wav
4,0.348098,-0.058533,-1.211667,-0.349726,-0.774786,-0.047239,0.425004,-0.266767,-1.092539,0.148986,0.404601,-0.142667,-0.185484,-0.494426,1,20000,0,8,A20000T00N08x04.wav


In [4]:
train_df['label'].value_counts()

,count
label,
1,40
2,13
3,13
4,11
5,8


#Process Data

##Drop Unnecessary columns

In [ ]:
columns = train_df.columns
columns

Index(['mfcc_0', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6',
       'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12',
       'spectral_rolloff', 'label', 'motor_code', 'transient_code',
       'noise_code', 'filename'],
      dtype='object')

In [ ]:
drop_column_list = ['motor_code', 'transient_code',
       'noise_code', 'filename']
updated_train_df = train_df.drop(drop_column_list, axis=1)
updated_train_df.head()

,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,spectral_rolloff,label
0,0.050365,0.009733,-1.055566,0.316141,0.669253,0.655374,-0.136285,0.004763,-1.181118,-0.990770,-0.350304,-0.880928,-0.515005,-0.626708,2
1,0.763096,-0.969426,-0.897181,-1.729356,-1.666303,-0.409425,1.724379,0.146817,0.466983,-0.349870,0.631503,0.174636,-0.639133,0.982519,1
2,0.761177,-0.606739,-0.410336,-0.881375,-0.883765,-1.129219,-0.685946,-1.338750,-0.162373,-0.273524,0.268508,0.113154,1.561239,0.788899,1
3,-0.258524,0.345973,1.287845,0.868921,-0.119996,1.767695,1.013693,0.791222,0.545691,-1.498075,-0.502849,0.491811,-1.149545,0.169105,4
4,0.348098,-0.058533,-1.211667,-0.349726,-0.774786,-0.047239,0.425004,-0.266767,-1.092539,0.148986,0.404601,-0.142667,-0.185484,-0.494426,1


In [ ]:
X = updated_train_df.drop('label', axis=1).values
Y = updated_train_df['label'].values

#Train

##Function for five-fold cross-validation

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
import numpy as np


def cross_validation(model, data = (X, Y), splits = 5):
    kf = KFold(n_splits=splits, shuffle=True, random_state=42)

    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []

    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]

        # Fit the defined model
        model.fit(X_train, y_train)

        # Make predictions on the test data
        y_pred = model.predict(X_valid)

        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_pred, y_valid))
        precision.append(precision_score(y_pred, y_valid, average = 'micro'))
        recall.append(recall_score(y_pred, y_valid, average = 'micro'))




        # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)

    print("Mean Accuracy: {}".format(accuracy_set.mean()))
    #print("Mean Precision: {}".format(precision_set.mean()))
    #print("Mean Recall: {}".format(recall_set.mean()))
    return accuracy_set.mean()




##Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# fit the logistic regression
accuracies = []
i_values = [50, 100, 150, 200, 250, 300]
for max_iter in i_values:
    print(f"Running Logistic Regression with max_iter = {max_iter}")
    lr = LogisticRegression(max_iter=max_iter)
    accuracy=cross_validation(lr)
    accuracies.append(accuracy)


Running Logistic Regression with max_iter = 50
Mean Accuracy: 0.8117647058823529
Running Logistic Regression with max_iter = 100
Mean Accuracy: 0.8117647058823529
Running Logistic Regression with max_iter = 150
Mean Accuracy: 0.8117647058823529
Running Logistic Regression with max_iter = 200
Mean Accuracy: 0.8117647058823529
Running Logistic Regression with max_iter = 250
Mean Accuracy: 0.8117647058823529
Running Logistic Regression with max_iter = 300
Mean Accuracy: 0.8117647058823529


In [ ]:
for accuracy in accuracies:
  print(accuracy)

0.8117647058823529
0.8117647058823529
0.8117647058823529
0.8117647058823529
0.8117647058823529
0.8117647058823529


##K-nearest-neighbor

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
accuracies = []
for k in range(1,15):
  print(k)
  model = KNeighborsClassifier(n_neighbors=k)
  accuracies.append(cross_validation(model))

1
Mean Accuracy: 0.8117647058823529
2
Mean Accuracy: 0.7411764705882352
3
Mean Accuracy: 0.7647058823529411
4
Mean Accuracy: 0.7529411764705881
5
Mean Accuracy: 0.6823529411764706
6
Mean Accuracy: 0.6823529411764706
7
Mean Accuracy: 0.6588235294117647
8
Mean Accuracy: 0.6470588235294118
9
Mean Accuracy: 0.6588235294117647
10
Mean Accuracy: 0.6470588235294118
11
Mean Accuracy: 0.6352941176470588
12
Mean Accuracy: 0.6588235294117647
13
Mean Accuracy: 0.6470588235294117
14
Mean Accuracy: 0.6470588235294117


In [ ]:
for accuracy in accuracies:
  print(accuracy)

0.8117647058823529
0.7411764705882352
0.7647058823529411
0.7529411764705881
0.6823529411764706
0.6823529411764706
0.6588235294117647
0.6470588235294118
0.6588235294117647
0.6470588235294118
0.6352941176470588
0.6588235294117647
0.6470588235294117
0.6470588235294117


## Random Forest

In [ ]:
# from sklearn.ensemble import RandomForestClassifier

# # Use 5-fold cross validation for hyper-parameter tuning
# # Try out different values and choose the best hyper-parameters
# rf = RandomForestClassifier()

# cross_validation(rf)
from sklearn.ensemble import RandomForestClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
accuracies = []
for n in range(10, 101, 10):
  temp = []
  for depth in range(1, 8):
    print("n_estimators =", n, ", max_depth =", depth)
    rf = RandomForestClassifier(n_estimators=n, max_depth=depth)
    acc = cross_validation(rf)
    temp.append(acc)
  accuracies.append(temp)

n_estimators = 10 , max_depth = 1
Mean Accuracy: 0.5058823529411764
n_estimators = 10 , max_depth = 2
Mean Accuracy: 0.6588235294117648
n_estimators = 10 , max_depth = 3
Mean Accuracy: 0.6823529411764706
n_estimators = 10 , max_depth = 4
Mean Accuracy: 0.7294117647058823
n_estimators = 10 , max_depth = 5
Mean Accuracy: 0.7176470588235294
n_estimators = 10 , max_depth = 6
Mean Accuracy: 0.7647058823529411
n_estimators = 10 , max_depth = 7
Mean Accuracy: 0.6941176470588235
n_estimators = 20 , max_depth = 1
Mean Accuracy: 0.5294117647058824
n_estimators = 20 , max_depth = 2
Mean Accuracy: 0.6117647058823529
n_estimators = 20 , max_depth = 3
Mean Accuracy: 0.6705882352941177
n_estimators = 20 , max_depth = 4
Mean Accuracy: 0.7294117647058823
n_estimators = 20 , max_depth = 5
Mean Accuracy: 0.7411764705882352
n_estimators = 20 , max_depth = 6
Mean Accuracy: 0.7176470588235294
n_estimators = 20 , max_depth = 7
Mean Accuracy: 0.8117647058823529
n_estimators = 30 , max_depth = 1
Mean Accuracy:

In [ ]:
for row in accuracies:
    print("\t".join([str(val) for val in row]))

0.5058823529411764	0.6588235294117648	0.6823529411764706	0.7294117647058823	0.7176470588235294	0.7647058823529411	0.6941176470588235
0.5294117647058824	0.6117647058823529	0.6705882352941177	0.7294117647058823	0.7411764705882352	0.7176470588235294	0.8117647058823529
0.5764705882352941	0.6470588235294117	0.7411764705882352	0.7647058823529411	0.7764705882352941	0.7882352941176471	0.776470588235294
0.5058823529411764	0.6470588235294118	0.7176470588235293	0.7529411764705882	0.8117647058823529	0.7176470588235294	0.7764705882352941
0.5411764705882353	0.6235294117647059	0.7764705882352941	0.7411764705882353	0.7647058823529411	0.7764705882352941	0.7529411764705882
0.5411764705882353	0.6823529411764706	0.7529411764705882	0.788235294117647	0.7882352941176469	0.776470588235294	0.776470588235294
0.5411764705882354	0.6470588235294118	0.7529411764705883	0.8117647058823529	0.7882352941176471	0.776470588235294	0.7647058823529412
0.5647058823529412	0.6470588235294118	0.7411764705882353	0.776470588235294

#Saving the best model

##Fit the best model

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
k=1
model = KNeighborsClassifier(n_neighbors=k)
model.fit(X,Y)


KNeighborsClassifier(n_neighbors=1)

In [ ]:
import joblib
model_path = '/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/models/KNNbestmodel.pkl'
joblib.dump(model, model_path)

['/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/models/KNNbestmodel.pkl']

## Save Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression
import joblib

lr_model = LogisticRegression(max_iter=100)

lr_model.fit(X, Y)

lr_model_path = '/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/models/LogisticRegression_bestmodel.pkl'

joblib.dump(lr_model, lr_model_path)

print(f"Logistic Regression model saved to: {lr_model_path}")

Logistic Regression model saved to: /content/drive/MyDrive/1:1_Arjan_Walia/Vessels/models/LogisticRegression_bestmodel.pkl


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib

rf_model = RandomForestClassifier(n_estimators=40, max_depth=6, random_state=42)

rf_model.fit(X, Y)

rf_model_path = '/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/models/RandomForest_bestmodel.pkl'

joblib.dump(rf_model, rf_model_path)

print(f"Random Forest model saved to: {rf_model_path}")

Random Forest model saved to: /content/drive/MyDrive/1:1_Arjan_Walia/Vessels/models/RandomForest_bestmodel.pkl
